# Chapter 14: Fine-tuning & Adaptation Techniques
## Notebook 03 — Advanced Adaptation: Instruction Tuning, Preference Data, Evaluation, Deployment

This notebook covers the parts of fine-tuning that go *beyond* SFT: **instruction tuning** at scale, **preference data** (RLHF and DPO), rigorous **evaluation** (held-out, win rates, LLM-as-judge), **catastrophic forgetting**, and **deployment** (registry, versioning) — the hand-off into Chapter 15 (MLOps).

### What you'll learn

| Topic | Section |
|-------|--------|
| Instruction tuning datasets (Alpaca format) | §2 |
| Preference data: RLHF and DPO | §3 |
| Evaluation: held-out, win-rate, LLM-as-judge | §4 |
| Catastrophic forgetting | §5 |
| Deployment, registry, versioning | §6 |
| Capstone design | §7 |

**Estimated time:** 2 hours

---
*Generated by Berta AI*

---
## 1. Setup

In [ ]:
import os, sys, json, math
from pathlib import Path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import config
from dataset_utils import load_jsonl, format_instruction
from training_utils import EvalHarness, exact_match, token_f1, win_rate_stub

np.random.seed(42)
print('DPO beta =', config.DPO_BETA)

---
## 2. Instruction Tuning Datasets (Alpaca-style)

**Alpaca format** has three fields per row: `instruction`, `input` (optional), and `output`. It became the de-facto schema after Stanford's Alpaca and is supported by virtually every fine-tuning framework.

Sources of high-quality instruction data:

- Curated human-written examples (smaller, highest quality).
- Self-instruct / model-generated then filtered (larger, must be filtered for quality and safety).
- Conversion from existing supervised tasks (e.g. classification → 'Classify the sentiment of: ...').

Quality > quantity. A few thousand carefully curated examples often beat a hundred thousand noisy ones.

In [ ]:
rows = load_jsonl(Path('..') / 'datasets' / 'instructions.jsonl')
df = pd.DataFrame(rows)
print(df.head(5).to_string(index=False))
print(f'\n{len(df)} instructions, {df["input"].astype(bool).sum()} with non-empty input.')

---
## 3. Preference Data: RLHF and DPO

SFT teaches the model *how to respond*; preference fine-tuning teaches it *which responses are preferred*. Each preference example has the form `(prompt, chosen, rejected)`.

### RLHF (Reinforcement Learning from Human Feedback)

Three stages: (1) SFT, (2) train a **reward model** on the preference pairs, (3) optimize the policy with PPO using the reward model. Powerful but operationally heavy: a separate reward model, KL-regularization, sensitive to hyperparameters.

### DPO (Direct Preference Optimization)

Rafailov et al. (2023) showed you can skip the reward model entirely. DPO directly optimizes the policy on preference pairs with a closed-form loss:

$$ \mathcal{L}_\text{DPO} = -\log \sigma\!\left( \beta \big[ (\log \pi_\theta(y_w|x) - \log \pi_\text{ref}(y_w|x)) - (\log \pi_\theta(y_l|x) - \log \pi_\text{ref}(y_l|x)) \big] \right) $$

where $y_w$ is the *winning* response, $y_l$ the *losing* one, $\pi_\theta$ the model we are training, and $\pi_\text{ref}$ a frozen reference (typically the SFT model). $\beta$ controls how strongly to deviate from the reference.

Below we implement the DPO loss in NumPy on toy logits.

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def dpo_loss(logp_chosen, logp_rejected, ref_logp_chosen, ref_logp_rejected, beta=0.1):
    """Toy DPO loss in NumPy on already-computed log-probs."""
    chosen_logratio = logp_chosen - ref_logp_chosen
    rejected_logratio = logp_rejected - ref_logp_rejected
    margin = beta * (chosen_logratio - rejected_logratio)
    loss = -np.log(sigmoid(margin) + 1e-12)
    return float(loss.mean()), {'margin': margin, 'chosen_logratio': chosen_logratio, 'rejected_logratio': rejected_logratio}

rng = np.random.default_rng(0)
n = 8
ref_lp_w = rng.uniform(-5, -1, size=n); ref_lp_l = rng.uniform(-6, -2, size=n)
# Case A: policy mirrors reference exactly (loss is large, no preference learned).
loss_a, _ = dpo_loss(ref_lp_w, ref_lp_l, ref_lp_w, ref_lp_l, beta=config.DPO_BETA)
# Case B: policy raises chosen and lowers rejected.
policy_lp_w = ref_lp_w + 0.5
policy_lp_l = ref_lp_l - 0.5
loss_b, info_b = dpo_loss(policy_lp_w, policy_lp_l, ref_lp_w, ref_lp_l, beta=config.DPO_BETA)
print(f'DPO loss when policy = reference: {loss_a:.4f}')
print(f'DPO loss when policy moves toward chosen: {loss_b:.4f}')
print('Per-example margin (positive means policy prefers chosen more than ref):')
print(np.round(info_b["margin"], 3))

**Reading the numbers**: DPO loss falls when the policy *raises* probability of `chosen` relative to the reference and *lowers* probability of `rejected`. The KL penalty implicit in the formula keeps the policy from drifting too far from the SFT model.

In [ ]:
# Inspect the chapter's preference dataset.
prefs = load_jsonl(Path('..') / 'datasets' / 'preferences.jsonl')
print(f'{len(prefs)} preference pairs')
print('Example:')
for k, v in prefs[0].items():
    print(f'  {k}: {v}')

---
## 4. Evaluation: Held-out, Win-rates, LLM-as-judge

After fine-tuning you must answer two questions:

1. **Did it get better at the target task?** Measured on a held-out set of the *same* distribution: exact match, F1, BLEU, ROUGE, accuracy.
2. **Did it regress on general capabilities?** Measured on broader benchmarks (MMLU, TruthfulQA, safety evals).

When the output is open-ended (chat, summaries), use **win rates** between two models judged by humans or another LLM. **LLM-as-judge** is fast and cheap but has caveats:

- Position bias (judges prefer A or B based on order). Always *swap and average*.
- Verbosity bias (longer answers win). Penalize length explicitly.
- Judge != target (a small judge can't score a frontier model).
- Judge alignment (the judge's preferences may not match users').

In [ ]:
eval_rows = load_jsonl(Path('..') / 'datasets' / 'eval_set.jsonl')
refs = [r['reference'] for r in eval_rows]

preds_baseline = [r['reference'].split()[0] if r['reference'] else '' for r in eval_rows]
preds_finetune = [r['reference'] for r in eval_rows]  # 'finetuned' returns the gold (toy)

h = EvalHarness(references=refs)
print('Baseline:', h.score(preds_baseline))
print('Fine-tuned:', h.score(preds_finetune))
print('Win rate FT vs baseline:', h.compare(preds_finetune, preds_baseline))

In [ ]:
# Position-bias check: swap A/B and average.
ab = win_rate_stub(preds_finetune, preds_baseline, refs)
ba = win_rate_stub(preds_baseline, preds_finetune, refs)
print('FT vs baseline:', ab)
print('baseline vs FT:', ba)
print('Symmetry check (should be approximately mirrored):',
      math.isclose(ab["a_win_rate"], ba["b_win_rate"]))

---
## 5. Catastrophic Forgetting

Fine-tuning moves model weights toward your data. If you push too hard, the model forgets what it knew before — *catastrophic forgetting*. Symptoms:

- Target metric improves, general benchmarks (MMLU, HellaSwag) fall.
- Out-of-distribution prompts produce nonsense.
- Refusal / safety behavior degrades.

Mitigations:

1. **PEFT** (LoRA, adapters) — frozen base preserves most knowledge.
2. **Lower learning rate** and fewer epochs.
3. **Mix in general data** (5–20% of your training mix from broad sources).
4. **KL regularization** to a reference model (DPO does this implicitly).
5. **Eval continuously** on broader benchmarks; stop early if regressions appear.

In [ ]:
# Toy 'forgetting' simulation: target accuracy rises, general accuracy falls.
epochs = np.arange(1, 11)
target_acc = 0.5 + 0.4 * (1 - np.exp(-0.4 * epochs))
general_acc = 0.78 - 0.05 * (epochs - 3).clip(min=0)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(epochs, target_acc, marker='o', label='target task acc')
ax.plot(epochs, general_acc, marker='s', label='general benchmark acc')
ax.axvline(epochs[np.argmax(target_acc - general_acc)], color='red', linestyle='--', label='early-stop point')
ax.set_xlabel('epoch'); ax.set_ylabel('accuracy'); ax.legend(); ax.set_title('Catastrophic forgetting trade-off')
plt.show()
print('Lesson: pick the checkpoint that maximizes target gain while general acc is still acceptable.')

---
## 6. Deployment: Model Registry, Versioning, Hand-off to MLOps

Every fine-tune produces an artifact plus metadata. A **model registry** stores:

- A unique **version** (semver or hash).
- The **base model** and **adapter** paths.
- The **dataset hash** (so you can reproduce or audit).
- **Hyperparameters** (rank, alpha, LR, epochs, beta for DPO).
- **Eval scores** on held-out and broader benchmarks.
- **Owner**, **created_at**, **status** (`staging` / `prod` / `retired`).

Below we build a tiny registry entry and write it as YAML — the same shape Chapter 15 will pick up for CD.

In [ ]:
import yaml, hashlib, datetime as dt

def dataset_hash(rows):
    text = json.dumps(rows, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(text).hexdigest()[:12]

entry = {
    'name': 'support-bot',
    'version': '1.2.0',
    'base_model': config.MODELS['base_lm'],
    'adapter_path': config.MODELS['lora_adapter'],
    'method': 'lora',
    'dataset_hash': dataset_hash(rows),
    'hyperparams': {
        'lora_rank': config.LORA_RANK,
        'lora_alpha': config.LORA_ALPHA,
        'lora_dropout': config.LORA_DROPOUT,
        'lr': config.LEARNING_RATE,
        'epochs': config.EPOCHS,
        'batch_size': config.BATCH_SIZE,
    },
    'eval': {
        'held_out_em': 0.71,
        'held_out_f1': 0.83,
        'win_rate_vs_baseline': 0.62,
        'mmlu_delta': -0.4,  # general benchmark regression in absolute %
    },
    'status': 'staging',
    'created_at': dt.date(2026, 5, 9).isoformat(),
    'owner': 'practitioner@berta.ai',
}
print(yaml.safe_dump(entry, sort_keys=False))

**Promotion gate** (Chapter 15 will automate this): a candidate moves from `staging` to `prod` only if held-out F1 increases by at least X, win rate is > 0.55, and `mmlu_delta` does not fall below -1.0.

---
## 7. Capstone Project Design

Build an **end-to-end fine-tuning project** in your domain. A workable scope for a week:

1. **Pick a task** with a clear input/output (support reply, code review, summarization with a fixed style).
2. **Curate 200–2000 examples**. Convert to Alpaca format. Hold out at least 10%.
3. **Run SFT** with LoRA (r=8, alpha=16) on a small open model. Track train/val loss.
4. **Optionally** collect 50–200 preference pairs and run **DPO** on top.
5. **Evaluate**: held-out EM/F1, win rate vs the base model, MMLU/TruthfulQA spot check.
6. **Register** the artifact (YAML/JSON) with all metadata above.
7. **Hand off** the registry to Chapter 15 for CI/CD and serving.

Anti-patterns to avoid:

- Training on the eval set (always hash and check).
- Optimizing only the target metric (catastrophic forgetting).
- Skipping a baseline (you need to know how much you actually gained).
- Single-judge LLM evaluation (always run position-swapped).

---
## 8. Key Takeaways

- **Instruction tuning** is the bread and butter of practical fine-tuning; format matters.
- **DPO** replaces RLHF for most preference fine-tuning — simpler, no reward model.
- **Evaluate broadly**, not just on the target. Watch for forgetting and safety regressions.
- **Versioned artifacts** with eval metadata are the unit of deployment.

Next — **Chapter 15: MLOps for AI Systems** — turn the registry into a real deployment pipeline.

---
*Generated by Berta AI*